# Automatic Encoding for the Iceberg Code 

We have seen in the [Steane code example](https://docs.quantinuum.com/guppy/ft/examples/steane_encoding.html) that Guppy FT provides automatic encoding for Guppy programs. In this example, we will demonstrate automatic encoding for the $[[k+2, k, 2]]$ Iceberg code. Unlike Steane, encoding programs with the Iceberg code requires the use of Nexus console - Quantinuum's cloud platform.

## Setup

As Nexus console is required, we will need the `qnexus` client package.

This additional Python package can be installed as follows
```shell
pip install qnexus
```

In addition, we will also need a Nexus console account. If you don't already have access to Nexus console, it is possible to apply for access at https://nexus.quantinuum.com/signup.

Once we have the `qnexus` package installed, we can log in to Nexus console and create a project to keep track of all of the jobs we run.

In [ ]:
import qnexus as qnx

qnx.login()

project = qnx.projects.get_or_create(name="iceberg_project")
qnx.context.set_active_project(project)

## Writing a computational program

We can now define our computational program using Guppy. Here we define a `prepare_ghz_state` function which prepares a GHZ state over $n$ qubits. We also need a `main` function which defines the entrypoint to our quantum program.

In [ ]:

from guppylang import guppy
from guppylang.std.quantum import qubit, h, cx, measure_array, collect_measurements
from guppylang.std.platform import output
from guppylang.defs import GuppyFunctionDefinition



def build_ghz_main(n_qubits: int) -> GuppyFunctionDefinition:
    @guppy
    def prepare_ghz_state() -> array[qubit, n_qubits]:
        qs = array(qubit() for _ in range(n_qubits))
        h(qs[0])
        for i in range(n_qubits - 1):
            cx(qs[i], qs[i + 1])
        return qs

    @guppy
    def main() -> None:
        ghz_qubits: array[qubit, n_qubits] = prepare_ghz_state()
        measurements = measure_array(ghz_qubits)
        output("m", collect_measurements(measurements))

    return main

We can now create a 16 qubit instance of our logical GHZ program and compile it to the HUGR IR.

In [ ]:
ghz_main = build_ghz_main(16)
pkg = ghz_main.compile()

## Defining Iceberg encoding parameters

We can now use the {py:class}`~guppyft.code.iceberg.encode.IcebergBuilder` to specify some key parameters of the Iceberg code. 

The {py:class}`~guppyft.code.iceberg.encode.IcebergBuilder` should be configured with an {py:class}`~guppyft.code.iceberg.encode.IcebergQEDPolicy` - a policy for when quantum error detection (QED) cycles can be applied. Guppy FT allows users to associate costs with different logical operations. These costs can then be accumulated as the program runs and a QED cycle is applied whenever a user defined threshold is exceeded.

In [ ]:
from guppyft.code.iceberg.encode import IcebergBuilder, IcebergQEDPolicy


qed_policy = IcebergQEDPolicy()

qed_policy.costs.cx_all = 1.0

# Insert a QED cycle after 10 CX gates are applied
qed_policy.threshold = 10

builder = IcebergBuilder().with_qed_policy(qed_policy)

Once we have defined a policy, we can use the {py:class}`~guppyft.code.iceberg.encode.IcebergBuilder` we have defined to annotate our compiled Guppy program with some metadata which stores all of the encoding parameters that we have specified. Computational to physical compilation can then be done serverside.  

In [4]:
# Specify the block size (number of logical qubits) and number of blocks
block_size = 2
n_blocks = 8
instance = builder.build(block_size=block_size, n_blocks=n_blocks)

# Annotate block_size, n_blocks, qed_policy and other parameters onto the compiled program.
instance.annotate(pkg) 

NameError: name 'builder' is not defined

## Uploading to Nexus console for execution

We can now upload our annotated GHZ program to the cloud using the `qnexus` client.

In [ ]:
import uuid

name = f"iceberg-nexus-{uuid.uuid4()}"
iceberg_hugr_ref = qnx.hugr.upload(hugr_package=pkg, name=name)

We can now define a configuration for the `Helios-1E` remote emulator. This emulator uses a physics informed noise model to emulate the noise present in the Quantinuum Helios device. 

In [ ]:
config = qnx.HeliosConfig(system_name="Helios-1E")

Finally, we can submit an execute job to run on the `Helios-1E` simulator by passing our compiled program to the `qnx.start_execute_job` method. Note how we have to specify the `max_cost` in HQCs and also specify a total number of physical qubits to be used in the simulation.

In [ ]:
ref_execute_job = qnx.start_execute_job(
    programs=[iceberg_hugr_ref],
    n_shots=[50],
    backend_config=config,
    max_cost=1000,
    name=f"Try Iceberg support GHZ-(n_logicals={16}, block_size={block_size})",
    n_qubits=(block_size + 2) * n_blocks + n_blocks,
)